In [1]:
# Ejercicio 1:
import requests

lista = ["Hulk", "Ironman", "Thor", "ana", "oso", "radar", "reconocer", "salas"]


def polindromos(lista):
    return list(map(es_polindromo, lista))

def es_polindromo(cadena):
    return cadena == cadena[::-1]

print(polindromos(lista))



[False, False, False, True, True, True, True, True]


In [2]:
# Ejercicio 2:
import requests

def cuadrados_sumados(n):
    return sum(map(lambda x: x**2, range(1, n+1)))

print(cuadrados_sumados(10))  

385


In [3]:
# Ejercicio 3:
import requests
from functools import reduce

def factorial(n):
    return reduce(lambda x, y: x * y, range(1, n + 1))
    


print(factorial(5))



120


In [11]:
# Ejercicio 4:

estudiantes = [
{"nombre": "Juan", "calificaciones": [7, 8, 6, 5]},
{"nombre": "Ana", "calificaciones": [9, 6, 8]},
{"nombre": "Pedro", "calificaciones": [4, 5, 3]},
{"nombre": "Maria", "calificaciones": [10, 9, 9]},
{"nombre": "Luis", "calificaciones": [3, 2, 5]}
]


def filtrar_aprobados(estudiantes):
    def promedio(calificaciones):
        return sum(calificaciones) / len(calificaciones)
    
    return list(filter(lambda estudiante: promedio(estudiante["calificaciones"]) >= 6, estudiantes))

print(filtrar_aprobados(estudiantes))


[{'nombre': 'Juan', 'calificaciones': [7, 8, 6, 5]}, {'nombre': 'Ana', 'calificaciones': [9, 6, 8]}, {'nombre': 'Maria', 'calificaciones': [10, 9, 9]}]


In [1]:
# Ejercicio 5(Largo):

import hashlib
import requests
from functools import reduce
from passlib.hash import nthash



HEADERS = {"User-Agent": "FunctionalChecker", "hibp-api-key": "000000000"}




hash_sha1 = lambda pwd: hashlib.sha1(pwd.encode('utf-8')).hexdigest().upper()
hash_ntlm = lambda pwd: nthash.hash(pwd).upper()


get_pwned_range = lambda prefix, mode: requests.get(
    f"https://api.pwnedpasswords.com/range/{prefix}?mode={mode}" if mode == "ntlm"
    else f"https://api.pwnedpasswords.com/range/{prefix}"
)

get_breaches = lambda domain: requests.get(
    f"https://haveibeenpwned.com/api/v3/breaches?Domain={domain}",
    headers=HEADERS
)

# Funciones de lógica funcional

def check_password(hash_function_callback, mode_string):
    """Comprueba si una contraseña ha sido filtrada."""
    try:
        pwd = input("Introduce la contraseña: ").strip()
        if not pwd:
            print("No se introdujo contraseña.")
            return

        hashed = hash_function_callback(pwd)
        prefix, suffix = hashed[:5], hashed[5:]

        response = get_pwned_range(prefix, mode_string)
        if response.status_code != 200:
            print("Error en la request.")
            return

        # Procesamos respuesta en modo funcional
        matches = list(
            filter(lambda line: line.split(':')[0] == suffix,
                   response.text.splitlines())
        )

        count = reduce(lambda acc, line: int(line.split(':')[1]) if line else acc, matches, 0)

        print(
            f"Resultado: La contraseña {'HA SIDO' if count else 'NO HA SIDO'} filtrada"
            f"{f' ({count} veces)' if count else ''}."
        )

    except Exception as e:
        print("Error en la comprobación:", e)


def check_domain():
    """Comprueba si un dominio ha sufrido filtraciones."""
    try:
        domain = input("Introduce el dominio (ej. gmail.com): ").strip()
        if not domain:
            print("No se introdujo dominio.")
            return

        response = get_breaches(domain)
        if response.status_code == 404:
            print("No hay filtraciones registradas para este dominio.")
            return
        elif response.status_code != 200:
            print(f"Error HTTP {response.status_code}")
            return

        data = response.json()
        print(f"\nFiltraciones encontradas para {domain}:")
        for b in data:
            print(f"• {b['Name']} – {b['BreachDate']} ({b['PwnCount']} cuentas)")
        print()

    except Exception as e:
        print("Error en la comprobación de dominio:", e)


# Main

if __name__ == "__main__":
    print("=== Have I Been Pwned – Checker ===")
    print("1️.  Comprobar contraseñas")
    print("2️.  Comprobar dominios")

    opcion = input("Elige modo (1/2): ").strip()

    if opcion == "1":
        print("\nModo contraseñas seleccionado.")
        print("a) SHA1 (estándar)\nb) NTLM (Windows)")
        tipo = input("Elige tipo de hash (a/b): ").strip().lower()

        # Callback según tipo de hash
        if tipo == "a":
            check_password(hash_sha1, "sha1")   
        elif tipo == "b":
            check_password(hash_ntlm, "ntlm")      
        else:
            print("Opción inválida.")

    elif opcion == "2":
        print("\nModo dominios seleccionado.")
        check_domain()

    else:
        print("Opción inválida.")


=== Have I Been Pwned – Checker ===
1️.  Comprobar contraseñas
2️.  Comprobar dominios

Modo contraseñas seleccionado.
a) SHA1 (estándar)
b) NTLM (Windows)
Resultado: La contraseña HA SIDO filtrada (3279 veces).
